In [9]:
from pathlib import Path
import pandas as pd

GTFS_DIR = Path("data/raw/gtfs_vmt/extracted")

print(GTFS_DIR.resolve())
print("Existiert:", GTFS_DIR.exists())

/Users/leon/Projects/transIT/data/raw/gtfs_vmt/extracted
Existiert: True


In [10]:
files = sorted(GTFS_DIR.glob("*"))

for file in files:
    print(f"{file.name:25} {file.stat().st_size / 1024 / 1024:8.2f} MB")

agency.txt                    0.00 MB
calendar.txt                  0.08 MB
calendar_dates.txt            1.06 MB
feed_info.txt                 0.00 MB
frequencies.txt               0.00 MB
routes.txt                    0.03 MB
shapes.txt                    0.00 MB
stop_times.txt              101.18 MB
stops.txt                     1.28 MB
transfers.txt                 1.85 MB
trips.txt                     6.62 MB


In [12]:
for file in sorted(GTFS_DIR.glob("*.txt")):
    if file.name == "stop_times.txt": #skippen weil zu groß
        print(f"{file.name:25}")
        continue

    with open(file, "r", encoding="utf-8-sig") as f:
        row_count = sum(1 for _ in f) - 1

    print(f"{file.name:25} {row_count:>12,} Rows")

agency.txt                          44 Rows
calendar.txt                     2,273 Rows
calendar_dates.txt              67,385 Rows
feed_info.txt                        1 Rows
frequencies.txt                      0 Rows
routes.txt                         857 Rows
shapes.txt                           0 Rows
stop_times.txt           
stops.txt                       13,169 Rows
transfers.txt                   28,132 Rows
trips.txt                      115,512 Rows


In [14]:
#Relevante Tabellen in Pandas anlegen
routes = pd.read_csv(
    GTFS_DIR / "routes.txt",
    dtype=str
)

stops = pd.read_csv(
    GTFS_DIR / "stops.txt",
    dtype=str
)

trips = pd.read_csv(
    GTFS_DIR / "trips.txt",
    dtype=str
)

agency = pd.read_csv(
    GTFS_DIR / "agency.txt",
    dtype=str
)

print("routes:", routes.shape)
print("stops:", stops.shape)
print("trips:", trips.shape)
print("agency:", agency.shape)

routes: (857, 8)
stops: (13169, 10)
trips: (115512, 10)
agency: (44, 6)


In [15]:
routes.head(5)

,route_id,agency_id,route_short_name,route_long_name,route_type,route_color,route_text_color,route_desc
0,9183_3,191,144_U,NaN,3,NaN,NaN,NaN
1,9182_3,191,142_U,NaN,3,NaN,NaN,NaN
2,9181_3,191,140_U,NaN,3,NaN,NaN,NaN
3,8047_3,231,77,NaN,3,NaN,NaN,NaN
4,7657_3,211,726,NaN,3,NaN,NaN,NaN


In [16]:
stops.head(5)

,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding,platform_code
0,000800164000,NaN,Ebing,NaN,50.001532000000,10.913847000000,0,NaN,0,NaN
1,de:06412:912:5:9,NaN,Frankfurt (Main) Süd,NaN,50.099130000000,8.686600000000,0,NaN,0,9
2,000800539150,NaN,Schney,NaN,50.166204000000,11.073624000000,0,NaN,0,NaN
3,de:16073:8010014:801001400:801001450,NaN,Bad Blankenburg (Thüringer Wald),NaN,50.682622000000,11.275307000000,0,NaN,0,3
4,de:15085:817::801012751:801012750,NaN,Gernrode (Harz),NaN,51.729730000000,11.149470000000,0,NaN,0,NaN


In [18]:
#VU im Datensatz --> 42 insgesamt
print("Verkehrsunternehmen:")
display(agency[["agency_id", "agency_name"]])

print("\nVerteilung der route_type-Werte:")
display(routes["route_type"].value_counts().sort_index())

Verkehrsunternehmen:


,agency_id,agency_name
0,41,Ilmenauer Omnibusverkehr
1,44,Omnibus Verkehrs Gesellschaft mbH Sonneberg/Thür.
2,46,MBB GmbH
3,73,Städtische Nahverkehrsgesellschaft mbH Suhl/Ze...
4,78,Deutsche Bahn
5,85,VWG Sömmerda
6,86,Werrabus
7,96,KomBus Verkehr GmbH (KomBus)
8,106,Süd-Thüringen-Bahn
9,111,GVB Verkehrs- und Betriebsgesellschaft Gera mb...



Verteilung der route_type-Werte:


route_type
0     33
2     42
3    782
Name: count, dtype: int64

In [19]:
#route_types
#0 --> 33 Tramlinien
#1 --> 0 U-Bahn-linien
#2 --> 42 Zuglinien
#3 --> 782 Buslinie

In [22]:
bus_routes = routes[routes["route_type"]=="3"].copy() #bustrips kopieren
display(bus_routes[["route_id", "agency_id", "route_text_color", "route_short_name", "route_long_name"]].head(20))

Bus: 782


,route_id,agency_id,route_text_color,route_short_name,route_long_name
0,9183_3,191,NaN,144_U,NaN
1,9182_3,191,NaN,142_U,NaN
2,9181_3,191,NaN,140_U,NaN
3,8047_3,231,NaN,77,NaN
4,7657_3,211,NaN,726,NaN
5,7656_3,218,NaN,188,NaN
6,7655_3,218,NaN,187,NaN
7,7654_3,218,NaN,186,NaN
8,7653_3,218,NaN,183,NaN
9,7652_3,218,NaN,181,NaN


In [24]:
#Buslinien zu VU joinen

bus_routes_with_agency = bus_routes.merge(
    agency[["agency_id", "agency_name"]],
    on="agency_id",
    how="left"
)

print("Anzahl Buslinien:", len(bus_routes_with_agency))

display(
    bus_routes_with_agency[
        ["agency_id", "agency_name"]
    ]
    .drop_duplicates()
    .sort_values("agency_name")
    .reset_index(drop=True)
)

Anzahl Buslinien: 782


,agency_id,agency_name
0,176,Abellio Rail Mitteldeutschland GmbH
1,237,Busbetrieb Piehler GmbH & Co. KG
2,78,Deutsche Bahn
3,131,EW Bus GmbH
4,172,Erfurter Bahn
5,119,Erfurter Verkehrsbetriebe AG (EVAG)
6,211,Firma Schieck
7,111,GVB Verkehrs- und Betriebsgesellschaft Gera mb...
8,166,Harzer Schmalspurbahnen GmbH
9,156,Ilchmann Tours GmbH


In [25]:
#Aktueller Erkenntnisstand: gering, sehr PTV-basics hier

In [27]:
agency_route_counts = (
    bus_routes_with_agency.groupby("agency_name").size())
display(agency_route_counts)

agency_name
Abellio Rail Mitteldeutschland GmbH                          6
Busbetrieb Piehler GmbH & Co. KG                             2
Deutsche Bahn                                                3
EW Bus GmbH                                                 37
Erfurter Bahn                                                9
Erfurter Verkehrsbetriebe AG (EVAG)                         27
Firma Schieck                                                1
GVB Verkehrs- und Betriebsgesellschaft Gera mbH (GVB)       17
Harzer Schmalspurbahnen GmbH                                 3
Ilchmann Tours GmbH                                          1
Ilmenauer Omnibusverkehr                                    35
JES Verkehrsgesellschaft mbH (JES)                          39
Jenaer Nahverkehr GmbH (JNV)                                19
KomBus Verkehr GmbH (KomBus)                                79
LWW Bustouristik GmbH                                        1
Lokale Nahverkehrsgesellschaft Fulda mbH   

In [32]:
#Endlich auf VMT filtern
vmt_bus_agencies = ["Erfurter Verkehrsbetriebe AG (EVAG)",
                    "GVB Verkehrs- und Betriebsgesellschaft Gera mbH (GVB)",
                    "JES Verkehrsgesellschaft mbH (JES)",
                    "Jenaer Nahverkehr GmbH (JNV)",     
                    "Stadtwirtschaft Weimar GmbH (SWG)",
                    "KomBus Verkehr GmbH (KomBus)",
                    "Personenverkehrsgesellschaft mbH Weimarer Land (PVGWL)",
                    "Verkehrsgemeinschaft Landkreis Gotha GbR (VLG)",
                    "Verkehrsunternehmen Andreas Schröder (VUS)"]

vmt_bus_routes = bus_routes_with_agency[
    bus_routes_with_agency["agency_name"].isin(vmt_bus_agencies)].copy()

print("Anzahl VMT-BusLinien:", len(vmt_bus_routes))

display(vmt_bus_routes.groupby("agency_name").size().sort_values(ascending=False))
                    

Anzahl VMT-BusLinien: 312


agency_name
KomBus Verkehr GmbH (KomBus)                              79
Personenverkehrsgesellschaft mbH Weimarer Land (PVGWL)    77
Verkehrsgemeinschaft Landkreis Gotha GbR (VLG)            43
JES Verkehrsgesellschaft mbH (JES)                        39
Erfurter Verkehrsbetriebe AG (EVAG)                       27
Jenaer Nahverkehr GmbH (JNV)                              19
GVB Verkehrs- und Betriebsgesellschaft Gera mbH (GVB)     17
Stadtwirtschaft Weimar GmbH (SWG)                          9
Verkehrsunternehmen Andreas Schröder (VUS)                 2
dtype: int64

In [49]:
#Relevante Routenmerkmale definieren
route_info = routes[
    ["route_id",
     "agency_id",
     "route_short_name",
     "route_long_name",
     "route_type",]].copy()

#Fahrten mit Linien joinen
trips_with_routes = trips.merge( #Jede Lineinroute genau zu ihrer Linie
    route_info,
    on="route_id",
    how="left",
    validate="many_to_one") 

#Dimensionen ausgeben --> 115512 Linienrouten mit 14 Linienroutenmerkmalen
rows = len(trips_with_routes)
col = len(trips_with_routes.columns)
print(f"({rows}, {col})")

display(
    trips_with_routes[
        ["trip_id", 
         "route_id",
         "agency_id",
         "route_short_name", #Linientext
         "direction_id",
         "trip_headsign"]].copy(10))

(115512, 14)


,trip_id,route_id,agency_id,route_short_name,direction_id,trip_headsign
0,40063150,9183_3,191,144_U,0,Kittelsthal WS
1,40063151,9183_3,191,144_U,0,Kittelsthal WS
2,40063149,9183_3,191,144_U,1,Wutha Farnroda Bhf.
3,40063148,9183_3,191,144_U,1,Wutha Farnroda Bhf.
4,40063145,9182_3,191,142_U,0,"Bad Tabarz üb.Seebach, Ruhla"
...,...,...,...,...,...,...
115507,38046728,6522_3,206,510,1,"Spechtsbrunn, Teich"
115508,38046729,6522_3,206,510,1,"Spechtsbrunn, Teich"
115509,38046735,6522_3,206,510,1,"Spechtsbrunn, Teich"
115510,38046731,6522_3,206,510,1,"Spechtsbrunn, Teich"


In [51]:
# Verkehrsunternehmen an die Fahrten anhängen
trips_with_routes = trips_with_routes.merge(
    agency[["agency_id", "agency_name"]],
    on="agency_id",
    how="left",
    validate="many_to_one"
)

print("Spalten:", trips_with_routes.columns.tolist())

display(
    trips_with_routes[
        [
            "trip_id",
            "route_short_name",
            "agency_id",
            "agency_name",
            "trip_headsign",
        ]
    ].head(10)
)

Spalten: ['route_id', 'service_id', 'trip_id', 'trip_headsign', 'trip_short_name', 'direction_id', 'block_id', 'shape_id', 'wheelchair_accessible', 'bikes_allowed', 'agency_id', 'route_short_name', 'route_long_name', 'route_type', 'agency_name']


,trip_id,route_short_name,agency_id,agency_name,trip_headsign
0,40063150,144_U,191,Verkehrsunternehmen Wartburgmobil VUW gkAöR,Kittelsthal WS
1,40063151,144_U,191,Verkehrsunternehmen Wartburgmobil VUW gkAöR,Kittelsthal WS
2,40063149,144_U,191,Verkehrsunternehmen Wartburgmobil VUW gkAöR,Wutha Farnroda Bhf.
3,40063148,144_U,191,Verkehrsunternehmen Wartburgmobil VUW gkAöR,Wutha Farnroda Bhf.
4,40063145,142_U,191,Verkehrsunternehmen Wartburgmobil VUW gkAöR,"Bad Tabarz üb.Seebach, Ruhla"
5,40063141,142_U,191,Verkehrsunternehmen Wartburgmobil VUW gkAöR,"Bad Tabarz üb.Seebach, Ruhla"
6,40063146,142_U,191,Verkehrsunternehmen Wartburgmobil VUW gkAöR,"Bad Tabarz üb.Seebach, Ruhla"
7,40063143,142_U,191,Verkehrsunternehmen Wartburgmobil VUW gkAöR,"Bad Tabarz üb.Seebach, Ruhla"
8,40063140,142_U,191,Verkehrsunternehmen Wartburgmobil VUW gkAöR,"Bad Tabarz üb.Seebach, Ruhla"
9,40063142,142_U,191,Verkehrsunternehmen Wartburgmobil VUW gkAöR,Wallfahrt/Rennsteig üb. Seeb


In [54]:
#Nach VMT-Fahrten filtern
vmt_bus_routes = trips_with_routes[
    trips_with_routes["agency_name"].isin(vmt_bus_agencies)].copy()

print("VMT Fahrten:", len(vmt_bus_routes))

display(
    vmt_bus_routes.groupby(["agency_name", "route_short_name"]).size().sort_values(ascending=False).head(15))

VMT Fahrten: 79888


agency_name                          route_short_name
Erfurter Verkehrsbetriebe AG (EVAG)  9                   4222
                                     6                   4028
KomBus Verkehr GmbH (KomBus)         A                   3489
                                     B                   2990
Erfurter Verkehrsbetriebe AG (EVAG)  1                   2142
                                     3                   2090
                                     90                  1971
                                     2                   1924
KomBus Verkehr GmbH (KomBus)         313                 1848
                                     302                 1820
Erfurter Verkehrsbetriebe AG (EVAG)  5                   1631
                                     51                  1590
                                     60                  1522
KomBus Verkehr GmbH (KomBus)         944                 1402
                                     215                 1381
dtype: int64

# VMT-Gebiet "selektieren"

In [56]:
stop_times = pd.read_csv(
    GTFS_DIR / "stop_times.txt",
    usecols=[
        "trip_id",
        "arrival_time",
        "departure_time",
        "stop_id",
        "stop_sequence",
    ],
    dtype={
        "trip_id": "string",
        "arrival_time": "string",
        "departure_time": "string",
        "stop_id": "string",
        "stop_sequence": "Int64",
    }
)

print("Zeilen:", len(stop_times))
print("Spalten:", len(stop_times.columns))
display(stop_times.head())

Zeilen: 1602175
Spalten: 5


,trip_id,stop_id,stop_sequence,arrival_time,departure_time
0,40063150,de:16063:163004::16300402,0,9:30:00,9:30:00
1,40063150,de:16063:163013::16301300,1,9:32:00,9:32:00
2,40063150,de:16063:163005::16300500,2,9:33:00,9:33:00
3,40063150,de:16063:163011::16301100,3,9:35:00,9:35:00
4,40063150,de:16063:1700754::170075400,4,9:38:00,9:38:00
